[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PerformanceEstimation/Tutorial-Europt-2026/blob/main/Part%201/Europt2026_Tutorial_Part1.ipynb)

# First hands-on session with PEPit : numerical examples of worst-case analyses

PEPit is a package enabling **computer-assisted worst-case analyses** of first-order optimization methods. The key underlying idea is to cast the problem of performing a worst-case analysis, often referred to as a performance estimation problem (PEP), as a semidefinite program (SDP) which can be solved numerically. For doing that, the package users are only required to write first-order methods nearly as they would have implemented them. The package then takes care of the SDP modelling parts, and the worst-case analysis is performed numerically via a standard solver.


PEPit can be installed following these [instructions](https://pypi.org/project/PEPit/), and a quickstart is available in its [documentation](https://pepit.readthedocs.io/en/latest/). More information can be found in the [PEPit reference paper](https://arxiv.org/pdf/2201.04040.pdf).

We refer to [this blog post](https://francisbach.com/computer-aided-analyses/) and the references therein for a more mathematical introduction to performance estimation problems.

## 1 Installing PEPit

In [ ]:
# If PEPit is not installed yet, you can run this cell.
!pip3 install pepit;

## 2 How to obtain a worst-case guarantee for gradient descent (GD) using PEPit

In this section, we provide a step-by-step tutorial on how to use PEPit to compute a worst-case guarantee for gradient descent.

That is, we consider the minimization problem

$$\min_{x} f(x),$$

where $f$ is an $L$-smooth convex function which is minimized at $x_*$. We denote by $x_n$ the output of gradient descent with step-size $\gamma$ after $n\in\mathbb{N}$ iterations, starting from $x_0$. We perform a worst-case analysis in the following sense: we compute the smallest possible value of $\tau(L,\gamma,n)$ such that

$$ f(x_{n})-f(x_*) \leqslant\  \tau(L,\gamma,n)\  \  \|x_0-x_*\|^2_2$$

is valid for all $L$-smooth convex function $f$ and initial iterate $x_0$. Computing the smallest possible such $\tau(L,\gamma,n)$ is equivalent to computing the worst-case value of $f(x_{n})-f(x_*)$ under the constraint $\| x_{0}-x_*\|_2^2\leqslant 1$.

PEPit formulates the problem of computing $\tau(L,\gamma,n)$ as an SDP which can be solved numerically. In what follows, we describe the few lines that a user has to provide for generating the numerical worst-case analysis of GD using PEPit.

In general, performing a worst-case analysis of a first-order method usually relies on **four main ingredients**:
- <font color='blue'>a class of problems</font> (containing the assumptions on the function to be minimized),
- <font color='red'>a first-order method</font> (to be analyzed),
- <font color='purple'>a performance measure</font> (measuring the quality of the output of the algorithm under consideration),
- <font color='green'>an initial condition</font> (measuring the quality of the initial iterate).

PEPit requires the user to specify a choice for each of those four elements.

### 2.1 Imports
Before anything else, we have to import the PEP and the classes of functions of interest.

Note: for smooth convex functions we use smooth strongly convex functions with $\mu=0$

In [ ]:
from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction

### 2.2 Initialization of PEPit
Then, we initialize a PEP object. This object allows manipulating the forthcoming ingredients of the PEP, such as functions and iterates.

In [ ]:
# Instantiate PEP
problem = PEP()

### 2.3 Choose parameters values

For the sake of the example, we pick some simple values for the problem class and algorithmic parameters, for which we perform the worst-case analysis below.

In [ ]:
# Specify values for the smoothness parameter L, number of iterations n, and the step size gamma.
L = 1
n = 4
gamma = 1 / L

### 2.4 Specifying the problem class
Next, we specify our assumptions on the <font color='blue'>**class of functions**</font> containing the function to be optimized, and instantiate a corresponding object. Here, we consider an $L$-smooth convex function.

<br> <font color='grey'>*Remark*: PEPit can handle various function classes</font> [(documentation)](https://pepit.readthedocs.io/en/latest/api/functions_and_operators.html).

Note: we use the ```SmoothStronglyConvexFunction``` class with $\mu=0$ here. Alternatively, one could use ```SmoothConvexFunction```.

In [ ]:
# Declare an L-smooth mu-strongly convex function named "func"
func = problem.declare_function(
                SmoothStronglyConvexFunction,
                mu=0,   # Strong convexity param.
                L=L)    # Smoothness param.

We also define $x_*$ as an optimal point and $f_*$ as the optimal value, as those are needed to express our performance criteria and initial condition.

In [ ]:
# Start by defining an optimal point xs = x_* and corresponding function value fs = f_*
xs = func.stationary_point()
fs = func(xs)

### 2.5 Algorithm initialization

Third, we can instantiate the starting points for the two gradient methods that we will run, and specify an <font color='green'>**initial condition**</font> on those points.
To this end, a starting point $x_0$ is introduced,  and a bound on the initial distance between those points is specified as $\|x_0-x_*\|^2\leqslant 1$.

<br> <font color='grey'> *Remark*: PEPit can handle various initial constraints, for example any linear combination of $f(x_0)-f_*$, $\|x_0-x_*\|^2$ or $\|\nabla f(x_0)\|^2$.</font>


In [ ]:
# Then define the starting point x0 of the algorithm
x0 = problem.set_initial_point()

# Set the initial constraint that is the distance between x0 and x^*
problem.set_initial_condition((x0 - xs) ** 2 <= 1)

### 2.6 Algorithm implementation

In this fourth step, we specify the <font color='red'>**algorithm**</font> in a natural format. In this example, we simply use the iterates (which are PEPit objects) as if we had to implement gradient descent in practice using a simple loop. Gradient descent with fixed step size $\gamma$ may be described as follows, for $t \in \{0,1, \ldots, n-1\}$
\begin{equation}
x_{t+1} = x_t - \gamma \nabla f(x_t).
\end{equation}

<br> <font color='grey'>*Remark*: PEPit can handle most first order algorithms that rely on some simple</font> [oracles.](https://pepit.readthedocs.io/en/latest/api/steps.html)<br><font color='grey'> More than 90 examples (for deterministic, stochastic, and/or composite optimization) are provided in the</font> [documentation](https://pepit.readthedocs.io/en/latest/examples.html).

In [ ]:
# Run n steps of gradient descent with step-size gamma
x = x0
for _ in range(n):
    x = x - gamma * func.gradient(x)

### 2.7 Setting up a performance measure

It is crucial for the worst-case analysis to specify the <font color='purple'>**performance metric**</font> for which we wish to compute a worst-case performance. In this example, we wish to compute the worst-case value of $f(x_n)-f_*$, which we specify as follows.

<br> <font color='grey'> *Remark*: PEPit can handle various performance metrics, for example any linear combination of $f(x_n)-f_*$, $\|x_n-x_*\|^2$ or $\|\nabla f(x_n)\|^2$.</font>


In [ ]:
# Set the performance metric to the function values accuracy
problem.set_performance_metric(func(x) - fs)

### 2.8 Solving the PEP

The last natural stage in the process is to solve the corresponding PEP. This is done via the following line, which will ask PEPit to perform the modelling steps and to call an appropriate SDP solver (which should be installed beforehand) to perform the worst-case analysis.

Solving the SDP is handled by the [CVXPY](https://www.cvxpy.org/) modelling layer.

The default CVXPY solver [SCS](https://web.stanford.edu/~boyd/papers/scs.html)  may not be the most appropriate  for solving SDPs; we advise using [CLARABEL](https://clarabel.org/) (free, open-source, available in colab) or [MOSEK](https://www.mosek.com/) (commercial, free academic license) if possible, for improved numerical performances).

In [ ]:
# Import the CVXPY namespace for solver selection
import cvxpy as cp
# Solve the PEP with verbose = 1 and CLARABEL solver
pepit_tau = problem.solve(verbose=1, solver=cp.CLARABEL)
print(f"\n\n*** Exact worst-case convergence bound of GD for {L}-smooth convex function after {n} steps is {pepit_tau}")

### 2.10 Using the built-in PEPit function ````wc_gradient_descent````

It is possible to reproduce the above results  using the built-in ````wc_gradient_descent```` function, provided as on of the 90+ examples in the package, see  [`GradientMethod`](https://pepit.readthedocs.io/en/latest/examples/a.html#gradient-descent).

For the parameters specified above, the worst-case guarantee obtained numerically with the built-in PEPit function ````wc_gradient_descent```` matches the analytical value of the best known (tight) worst-case guarantee, which is also provided by the PEPit function.

That **analytical value** is provided in [1, Theorem 1]: when $\gamma \leqslant \frac{1}{L}$, the exact (tight) worst case guarantee is given by:
$$f(x_n)-f_* \leqslant \frac{L||x_0-x_\star||^2}{4nL\gamma+2}$$
which is achieved on some Huber loss functions.


[1] Y. Drori, M. Teboulle (2014). [Performance of first-order methods for smooth convex minimization: a novel approach](https://arxiv.org/pdf/1206.3209.pdf). Mathematical Programming 145(1–2), 451–482.


In [ ]:
from PEPit.examples.unconstrained_convex_minimization import wc_gradient_descent
pepit_tau, theoretical_tau = wc_gradient_descent(L=L, gamma=1 / L, n=4, verbose=1)

#### 2.11 Plotting worst-case guarantees as functions of number of steps

In [ ]:
import numpy as np
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize

# Set a list of numner of steps
ns = range(25)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
theoretical_taus = list()
for n in ns:
    pepit_tau, theoretical_tau = wc_gradient_descent(           L=L,
                                                                 gamma=gamma,
                                                                 n=n,
                                                                 verbose=0,
                                                                )
    pepit_taus.append(pepit_tau)
    theoretical_taus.append(theoretical_tau)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
import matplotlib.pyplot as plt
plt.plot(ns, theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps')
plt.ylabel('Worst-case guarantee')

plt.show()

inverse_pepit_taus = [1/tau for tau in pepit_taus]
inverse_theoretical_taus = [1/tau for tau in theoretical_taus]

plt.plot(ns, inverse_theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(ns, inverse_pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps')
plt.ylabel('Inverse worst-case guarantee')

plt.show()


#### 2.12 Plotting worst-case guarantees as functions of the step size

In [ ]:
import numpy as np
# Set the parameters
n = 1      # iteration counter
L = 1      # smoothness parameter

# Set a list of step-sizes to test
gammas = np.linspace(0, 2 / L, 41)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
theoretical_taus = list()
for gamma in gammas:
    pepit_tau, theoretical_tau = wc_gradient_descent(           L=L,
                                                                 gamma=gamma,
                                                                 n=n,
                                                                 verbose=0,
                                                                )
    pepit_taus.append(pepit_tau)
    theoretical_taus.append(theoretical_tau)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the step-size
import matplotlib.pyplot as plt
plt.plot(gammas, theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(gammas, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Step-size')
plt.ylabel('Worst-case guarantee')

plt.show()

### Question: what is happening here with the theoretical bound?

Your turn!

## 3 Two-step gradient descent

Let us now consider the following optimization setup: we want to minimize a smooth strongly convex function
$$ \min_{x\in\mathbb{R}^d} f(x),$$
but using a multistep gradient algorithm. That is, we will iterate:
\begin{align*}
x_{1} &= x_0 - \gamma_1 \nabla f(x_0),\\
x_{2} &= x_1 - \gamma_2 \nabla f(x_1),
\end{align*}
and our goal will be to choose a pair $(\gamma_1, \gamma_2)$. We proceed with the following choice:

$$\min_{\gamma_1,\gamma_2}\,\, \max_{f\in\mathcal{F}_{\mu,L}} \left\{ \frac{\|x_2-x_\star\|^2_2}{\|x_0-x_\star\|^2_2} \, : \, x_2 \text{ obtained from } x_{1} = x_0 - \gamma_1 \nabla f(x_0),\,x_{2} = x_1 - \gamma_2 \nabla f(x_1)  \right\}.$$


To do this, we will perform 2D grid searches, and observe the numerics.

For simplicity, let us again carry out the analysis for specific parameter values. 

In [ ]:
L, mu = 1, .1

... and import the appropriate function class!

In [ ]:
# import PEPit and the required function class
from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction

# import numpy and matplotlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

### 3.1 Numerical exploration

The following PEPit code is standard and allows evaluating the performance of a sequence of stepsizes.

In [ ]:
def wc_gradient_descent(L, mu, gammas, verbose = 0):
    
    n = len(gammas)
    
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth convex function
    func = problem.declare_function(SmoothStronglyConvexFunction, L=L, mu=mu)

    # Start by defining its unique optimal point xs = x_* and corresponding function value fs = f_*
    xs = func.stationary_point(name='xs')

    # Then define the starting point x0 of the algorithm
    x0 = problem.set_initial_point(name='x0')

    # Set the initial constraint that is the distance between x0 and x^*
    problem.set_initial_condition((x0 - xs) ** 2 <= 1)

    # Run n steps of the GD method
    x = x0
    for i in range(n):
        x = x - gammas[i] * func.gradient(x)
        x.set_name('x{}'.format(i+1))

    # Set the performance metric to the function values accuracy
    problem.set_performance_metric( (x - xs) ** 2 )

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose)

    # Return the worst-case guarantee of the evaluated method together with the computed dual values and dual slack matrix S ("residual"):
    return pepit_tau

The following routine implements grid search for the stepsizes.

In [ ]:
def pepit_grid_search(gamma1_grid, gamma2_grid, mu, L):
    """
    Evaluate wc_gradient_descent on a grid of (gamma1, gamma2).

    Parameters
    ----------
    gamma1_grid : array-like
        Grid values for gamma1.
    gamma2_grid : array-like
        Grid values for gamma2.
    mu, L : float
        Problem parameters.

    Returns
    -------
    G1, G2 : ndarray
        Meshgrid of gamma1 and gamma2.
    tau_grid : ndarray
        Grid of pepit_tau values.
    """

    G1, G2 = np.meshgrid(gamma1_grid, gamma2_grid)
    tau_grid = np.zeros_like(G1)

    total = tau_grid.size

    for k, (i, j) in enumerate(np.ndindex(G1.shape), start=1):
        gamma1 = G1[i, j]
        gamma2 = G2[i, j]

        pepit_tau = wc_gradient_descent(mu=mu, L=L, gammas=[gamma1, gamma2])
        tau_grid[i, j] = pepit_tau

        print(f'{k} / {total} grid points computed', end='\r', flush=True)

    return G1, G2, tau_grid

def plot_pepit_landscape(G1, G2, tau_grid):
    """
    Plot the contour landscape of pepit_tau and highlight the minimum.

    Parameters
    ----------
    G1, G2 : ndarray
        Meshgrid of gamma1 and gamma2.
    tau_grid : ndarray
        Grid of pepit_tau values.

    Returns
    -------
    i_best, j_best : int
        Indices of the best grid point.
    gamma1_best, gamma2_best : float
        Best step sizes.
    tau_best : float
        Best value of pepit_tau.
    """

    plt.figure()

    levels = np.logspace(
        np.log10(np.nanmin(tau_grid)),
        np.log10(np.nanmax(tau_grid)),
        50
    )

    cf = plt.contourf(
        G1, G2, tau_grid,
        levels=levels,
        cmap='viridis_r',
        norm=LogNorm()
    )

    plt.colorbar(cf)

    # locate minimum
    idx = np.nanargmin(tau_grid)
    i_best, j_best = np.unravel_index(idx, tau_grid.shape)

    gamma1_best = G1[i_best, j_best]
    gamma2_best = G2[i_best, j_best]
    tau_best = tau_grid[i_best, j_best]

    # mark minimum
    plt.scatter(gamma1_best, gamma2_best, s=120, color='red',
                edgecolor='black', zorder=3)

    plt.xlabel(r'$\gamma_1$')
    plt.ylabel(r'$\gamma_2$')
    plt.title(r'Worst-case $\frac{\|x_2-x_\star\|^2}{\|x_0-x_\star\|^2}$')

    plt.show()

    print(f"Best grid index: (i,j) = ({i_best}, {j_best})")
    print(f"gamma1 = {gamma1_best:.4f}, gamma2 = {gamma2_best:.4f}")
    print(f"tau = {tau_best:.4e}")

    return i_best, j_best, gamma1_best, gamma2_best, tau_best

Using the previous pieces of code, perform a grid search over the step sizes and plot the corresponding convergence rate values.
Don't be too greedy (the grid search cost quickly becomes too high) and try to get to a reasonable approximation of the optimal pair $(\gamma_1,\gamma_2)$.


In [ ]:
# plot (2-step): parameterize the grid for the 2D grid search (
# grid parameters:
gamma_min, gamma_max = 0.1, 3.2 #TODO OPERAND
nb_gammas = 30 # do not make this too large :)

# grid search:
gamma1_vals = np.linspace(gamma_min, gamma_max, nb_gammas)
gamma2_vals = gamma1_vals
G1, G2, tau_grid = pepit_grid_search(gamma1_vals, gamma2_vals, mu=mu, L=L)

In [ ]:
_, _, gamma1_best, gamma2_best, tau_best = plot_pepit_landscape(G1, G2, tau_grid)

As we want to find properties of the optimal point, let us refine the grid around the target point (possibly several times)!

In [ ]:
gamma1_min, gamma1_max = 1.383, 1.384 #TODO OPERAND
gamma2_min, gamma2_max = 2.650, 2.651 #TODO OPERAND
nb_gammas = 10

gamma1_vals = np.linspace(gamma1_min, gamma1_max, nb_gammas)
gamma2_vals = np.linspace(gamma2_min, gamma2_max, nb_gammas)

G1, G2, tau_grid = pepit_grid_search(gamma1_vals, gamma2_vals, mu=mu, L=L)
_, _, gamma1_best, gamma2_best, tau_best = plot_pepit_landscape(G1, G2, tau_grid)

How do you summarize your findings?

### 3.2 Comparison with silver schedules ($n=2$)

Compare your numerically discovered stepsizes with the silver schedule for $n=2$ (see, [5])
\begin{align*}
    \gamma_1 & = \frac{1}{L}\left(\frac{\sqrt{1 + (1-\kappa)^2} - \kappa}{1-\kappa}\right)\\
    \gamma_2 & = \frac{1}{L}\left(\frac{\sqrt{1 + (1-\kappa)^2}+2+\kappa}{1+3\kappa}\right),
\end{align*}
with $\kappa=\tfrac{\mu}{L}$.

In [ ]:
kappa = mu/L
gammas = [0, 0] # Your choice here!
pepit_tau = wc_gradient_descent(L, mu, gammas, verbose = 0)

print('The best tau observed numerically: {}, vs. silver schedule: {}'.format(tau_best,pepit_tau))

What do you conclude?

### 3.3 Try this pattern on a simple quadratic problem and compare it to vanilla gradient descent

In [ ]:
L, mu = 1., 0.1
kappa = mu / L
H = np.array([L, mu])
grad = lambda x: H * x
f = lambda x: 0.5 * (H * x**2).sum()

def run_gd(x0, T):
    x, traj = x0.copy(), [x0.copy()]
    gamma = 2 / (L + mu)
    for _ in range(T):
        x -= gamma * grad(x)
        traj.append(x.copy())
    return np.array(traj)

def run_silver(x0, T):
    x, traj = x0.copy(), [x0.copy()]
    gamma1 = (np.sqrt(1 + (1-kappa)**2) - kappa) / (L * (1 - kappa)) #TODO OPERAND
    gamma2 = (np.sqrt(1 + (1-kappa)**2) + 2 + kappa) / (L * (1 + 3*kappa)) #TODO OPERAND
    for _ in range(T):
        x -= gamma1 * grad(x); traj.append(x.copy())
        x -= gamma2 * grad(x); traj.append(x.copy())
    return np.array(traj)

x0 = np.array([-2.5, 1.5])
traj_gd     = run_gd(x0, T=20)
traj_silver = run_silver(x0, T=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

X, Y = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-2, 2, 200))
ax1.contour(X, Y, 0.5*(H[0]*X**2 + H[1]*Y**2), levels=15, cmap="Blues")
for traj, color, label in [(traj_gd, "tomato", "GD"), (traj_silver, "mediumpurple", "Silver")]:
    ax1.plot(traj[:, 0], traj[:, 1], "o-", color=color, ms=3, label=label)
ax1.plot(*x0, "go", ms=8, label="start")
ax1.plot(0, 0, "*", color="gold", ms=12, label="x*")
ax1.set(title="Trajectory", xlabel="x1", ylabel="x2"); ax1.legend()

ax2.semilogy([f(x) for x in traj_gd],     "o-", color="tomato",       ms=3, label="GD")
ax2.semilogy([f(x) for x in traj_silver],  "o-", color="mediumpurple", ms=3, label="Silver")
ax2.set(title="Excess loss", xlabel="iteration", ylabel="f(xk) - f*"); ax2.legend()

plt.tight_layout(); plt.show()

### Going further <a class="anchor" id="sec4"></a>

More complete references and demos can be found in [PEPit's documentation and demos](https://github.com/PerformanceEstimation/PEPit/blob/master/ressources/demo).
- Algebraic techniques for solving linear matrix inequalities; see e.g., [4].
- Optimize over richer families of algorithms (e.g., families that include momentum?), see, e.g., [1, 3, 6, 7].
- Algorithm design based on nonlinear optimization; see, for instance, [6, 7].

---

### References

[1] Y. Drori and M. Teboulle, *Performance of first-order methods for smooth convex minimization: a novel approach*, Mathematical Programming, 2014. [arXiv:1206.3209](https://arxiv.org/abs/1206.3209)

[2] A. B. Taylor, J. M. Hendrickx, and F. Glineur, *Smooth strongly convex interpolation and exact worst-case performance of first-order methods*, Mathematical Programming, 2017. [arXiv:1502.05666](https://arxiv.org/abs/1502.05666)

[3] D. Kim and J. A. Fessler, *Optimized first-order methods for smooth convex minimization*, Mathematical Programming, 2016. [arXiv:1406.5468](https://arxiv.org/abs/1406.5468)

[4] S. Naldi, M.S. El Din, A. Taylor, W. Wang, *Solving parametric linear matrix inequalities*, 2025. [HAL:05505762](https://hal.sorbonne-universite.fr/hal-05505762/document)

[5] J. M. Altschuler and P. A. Parrilo, *Acceleration by Stepsize Hedging I: Multi-Step Descent and the Silver Stepsize Schedule*, Journal of the ACM, 2025. [arXiv:2309.07879](https://arxiv.org/abs/2309.07879)

[6] S. Das Gupta, B. P. G. Van Parys, and E. K. Ryu, *Branch-and-bound performance estimation programming: a unified methodology for constructing optimal optimization methods*, Mathematical Programming, 2024. [arXiv:2203.07305](https://arxiv.org/abs/2203.07305)

[7] Y. Kamri, J. M. Hendrickx, and F. Glineur, *Numerical Design of Optimized First-Order Algorithms*, 2025. [arXiv:2507.20773](https://arxiv.org/abs/2507.20773)
